# Selector 5 evaluation on the A100

This notebook first freezes the selector-5 target ranking from the cached RobustRAG responses, then evaluates the selected attacks with live Mistral-7B inference on the A100. Selection uses the cached responses; the live evaluation is the GPU workload. Files and results stay local to the server—no rclone or Google Drive setup.

In [ ]:
from getpass import getuser
from pathlib import Path

PROJECT_DIR = Path.home() / "rag-a100" / "project"
SOURCE_OWNER = "connor"
SOURCE_RUN_NAME = "table1-current_v2-open_nq-20261109T021501Z"
DATASET = "open_nq"
OUTPUT_OWNER = getuser()
TARGET_SELECTOR = "5"
DOCUMENT_GENERATOR = "delete_gold_segments"
MAX_CANDIDATES = "1"

In [ ]:
import json
import os
import subprocess
import sys

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR / "data" / "robustrag"
SOURCE_RUN = (
    PROJECT_DIR / "outputs" / SOURCE_OWNER / "robustrag-reproduction" / SOURCE_RUN_NAME
)
RUNNER_DIR = PROJECT_DIR / ".robustrag-runner"
OUTPUT_ROOT = (
    PROJECT_DIR / "outputs" / OUTPUT_OWNER / "promotion-attack" / f"selector5-{DATASET}"
)
SELECTION_ROOT = OUTPUT_ROOT / "selection"
LIVE_ROOT = OUTPUT_ROOT / "live"
NLTK_DATA_DIR = PROJECT_DIR / ".nltk-data"
dataset_file = DATA_DIR / f"{DATASET}.json"
RUN_ENV = dict(
    os.environ,
    NLTK_DATA=str(NLTK_DATA_DIR),
    PYTHONPATH=str(PROJECT_DIR),
    PYTHONHASHSEED="0",
)

required_paths = (
    PROJECT_DIR / "reproduction" / "robustrag_setup.py",
    PROJECT_DIR / "scripts" / "run_promotion_selection.py",
    PROJECT_DIR / "scripts" / "run_promotion_live.py",
    PROJECT_DIR / "third_party" / "robustrag_upstream",
    SOURCE_RUN / "manifest.json",
)
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing server files: " + ", ".join(map(str, missing_paths)))

source_manifest = json.loads((SOURCE_RUN / "manifest.json").read_text())
if (source_manifest.get("status"), source_manifest.get("profile"), source_manifest.get("dataset")) != (
    "complete", "current_v2", DATASET
):
    raise ValueError(f"Unexpected reproduction source manifest: {SOURCE_RUN / 'manifest.json'}")

print("Notebook Python:", sys.executable)
print("Source run:", SOURCE_RUN)
print("Local output:", OUTPUT_ROOT)

## Prepare the Python 3.11 CUDA environment and local inputs

The notebook kernel can be any working remote Python. Experiment commands run in a separate Python 3.11 environment with CUDA-enabled PyTorch. Dependencies follow the existing A100 notebook setup; the old `litellm` dependency is not needed because the pinned runner defers those imports. The source run must already be copied into the local `outputs/` path above.

In [ ]:
def run_checked(args, **kwargs):
    args = [str(arg) for arg in args]
    completed = subprocess.run(args, check=False, **kwargs)
    if completed.returncode:
        if getattr(completed, "stdout", None):
            print(completed.stdout, end="")
        if getattr(completed, "stderr", None):
            print(completed.stderr, end="", file=sys.stderr)
        raise RuntimeError(f"Command failed with exit status {completed.returncode}: {' '.join(args)}")
    return completed

run_checked([sys.executable, "-m", "pip", "install", "-q", "uv==0.13.5"])
UV = [sys.executable, "-m", "uv"]
ENV_DIR = PROJECT_DIR / ".robustrag-py311"
PYTHON = ENV_DIR / "bin" / "python"
if not PYTHON.exists():
    run_checked([*UV, "venv", "--python", "3.11", "--managed-python", ENV_DIR])

# Install a CUDA 12.1 build so the live stage actually uses the A100.
run_checked([
    *UV, "pip", "install", "--reinstall", "--python", PYTHON,
    "--index-url", "https://download.pytorch.org/whl/cu121",
    "--extra-index-url", "https://pypi.org/simple",
    "torch==2.2.1",
])
SPACY_MODEL = (
    "https://github.com/explosion/spacy-models/releases/download/"
    "en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
)
run_checked([
    *UV, "pip", "install", "--python", PYTHON,
    "numpy==1.26.4", "transformers==4.40.1", "tokenizers==0.19.1",
    "accelerate==0.29.3", "huggingface-hub==0.23.5", "sentencepiece==0.2.0",
    "joblib==1.3.2", "nltk==3.8.1", "spacy==3.7.4", "tqdm==4.66.1",
    "setuptools<81", SPACY_MODEL,
])

RUN_ENV["NLTK_DATA"] = str(NLTK_DATA_DIR)
run_checked([
    PYTHON, "-m", "nltk.downloader", "-d", NLTK_DATA_DIR, "stopwords"
], env=RUN_ENV)
preflight_code = (
    "import json, torch, spacy, transformers, joblib; "
    "from nltk.corpus import stopwords; "
    "print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, "
    "'available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) "
    "if torch.cuda.is_available() else None})); "
    "assert torch.cuda.is_available(), 'CUDA is unavailable in this Python environment'; "
    "assert 'A100' in torch.cuda.get_device_name(0), 'Expected an A100 GPU'; "
    "assert 'the' in stopwords.words('english'); "
    "assert spacy.load('en_core_web_sm')('Paris')[0].pos_"
)
preflight = run_checked(
    [PYTHON, "-c", preflight_code], env=RUN_ENV, capture_output=True, text=True
)
print("A100 environment:", preflight.stdout.strip())

In [ ]:
from reproduction.robustrag_setup import build_runner, ensure_datasets

build_runner(PROJECT_DIR / "third_party" / "robustrag_upstream", RUNNER_DIR)
ensure_datasets(DATA_DIR, (DATASET,))
SELECTION_ROOT.mkdir(parents=True, exist_ok=True)
LIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Verified dataset:", dataset_file)

## Freeze selector-5 targets

This CPU-side step deterministically picks selector-5 targets from the cached responses. It is not the GPU evaluation; the next stage runs the selected attacks with Mistral on the A100.

In [ ]:
selection_command = [
    PYTHON, "-u", "-m", "scripts.run_promotion_selection",
    "--source-run", SOURCE_RUN,
    "--dataset-path", dataset_file,
    "--runner-dir", RUNNER_DIR,
    "--output-root", SELECTION_ROOT,
    "--target-selector", TARGET_SELECTOR,
    "--document-generator", DOCUMENT_GENERATOR,
]
result = run_checked(
    selection_command, cwd=PROJECT_DIR, env=RUN_ENV,
    capture_output=True, text=True,
)
print(result.stdout, end="")
result_line = next(
    line for line in reversed(result.stdout.splitlines())
    if line.startswith("PROMOTION_SELECTION_RESULT=")
)
selection_summary = json.loads(result_line.split("=", 1)[1])
SELECTION_RUN = Path(selection_summary["output_dir"])
print("Frozen selection:", SELECTION_RUN)

## Evaluate on the A100 with live Mistral inference

Runs the selected top-1 attack for the source run's full recorded sample limit. The source reproduction was recorded on an L4; the live manifest records the A100 hardware mismatch. Mistral inference can take time and download model weights on its first run.

In [ ]:
live_command = [
    PYTHON, "-u", "-m", "scripts.run_promotion_live",
    "--selection-run", SELECTION_RUN,
    "--dataset-path", dataset_file,
    "--runner-dir", RUNNER_DIR,
    "--output-root", LIVE_ROOT,
    "--limit", str(source_manifest["sample_limit"]),
    "--max-candidates", MAX_CANDIDATES,
    "--allow-gpu-mismatch",
]
run_checked(live_command, cwd=PROJECT_DIR, env=RUN_ENV)

In [ ]:
print("Selection output:", SELECTION_RUN)
print("Live A100 evaluation output:", LIVE_ROOT)
print("Completed live runs:", [path.name for path in sorted(LIVE_ROOT.glob("promotion-live-*"))])